In [1]:
import pandas as pd
import numpy as np
import altair as alt

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegressionCV, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.multiclass import OneVsRestClassifier
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC

In [ ]:
track_history  = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/now_playing.csv")
track_analysis = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/track_analysis.csv")
track_metadata = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/track_metadata.csv")
weather        = pd.read_csv("https://raw.githubusercontent.com/jdmremi/kcpr/refs/heads/data/weather.csv")

# convert all request times to datetime formats
for df in [track_history, track_analysis, track_metadata, weather]:
    df["request_time"] = pd.to_datetime(df["request_time"])

# converts a string in the format mm:ss to milliseconds
def mm_ss_to_ms(mm_ss_str):
    minutes, seconds = map(int, mm_ss_str.split(":"))
    return ((minutes * 60) + seconds) * 1000

# remove analysis/metadata observations whose id is null
track_analysis = track_analysis[~track_analysis["id"].isna()]
track_metadata = track_metadata[~track_metadata["id"].isna()]
# remove all metadata rows whose genres are null
track_metadata = track_metadata.dropna(subset=["genres"])
# if any values are missing in the track metadata (which is acceptable), set them to be an empty string
track_metadata.fillna("", inplace=True)
# change track analysis duration (mm:ss) to milliseconds
track_analysis["duration"] = track_analysis["duration"].apply(lambda d: mm_ss_to_ms(d))
# remove dB from track analysis loudness and convert to an int (e.g., "-2 dB" becomes -2)
track_analysis["loudness"] = track_analysis["loudness"].apply(lambda db: db.replace(" dB", "")).astype(np.int16)
# split camelot into three columns: X/y position via cos(camelot num * 2pi/12) and is_major
track_analysis["is_major"] = (track_analysis["camelot"].apply(lambda camelot: "B" in camelot)).astype(np.int16)
# turn camelot into coordinates (better for modeling; see explanation below)
track_analysis["camelot_x"] = np.cos(track_analysis["camelot"].str.extract(r'(\d+)').astype(np.int16) * 2 * np.pi / 12)
track_analysis["camelot_y"] = np.sin(track_analysis["camelot"].str.extract(r'(\d+)').astype(np.int16) * 2 * np.pi / 12)

# keep only relevant features
track_analysis = track_analysis[["request_id", "request_time", "mode", "camelot", "camelot_x", "camelot_y", "is_major", "duration", "tempo", "popularity", "energy", "happiness", "acousticness", "danceability", "instrumentalness", "speechiness", "liveness", "loudness"]]
track_history  = track_history[["request_id", "request_time", "title", "artist", "album", "duration", "program_start_ts", "plays", "is_ad"]]
track_metadata = track_metadata.drop(columns=["id", "artist_sort_name", "release_statuses", "release_languages", "release_scripts", "release_qualities"])

# separate ads and actual songs played into separate dataframes
track_ad_history = track_history[track_history["is_ad"]]
track_history    = track_history[~track_history["is_ad"]]

# combine tables
data = pd.merge(track_analysis, track_metadata, on=["request_id", "request_time"], how="inner")
data = pd.merge(data, track_history, on=["request_id", "request_time"], how="inner")
# data = pd.merge(data.drop("genres", axis=1), track_genres, on=["request_id", "request_time"], how="inner")

The camelot wheel is a circle, so standard linear numbers can confuse ML models. Specifically, a model might think 12 and 1 are far apart, even though they are neighbors. The solution used here is projecting the 12 positions onto a 2D coordinate system using cos/sin.

For example, 12 and 1 on the camelot wheel are in key (and should therefore have a small "distance"). 12 lies at point (1, 0); 1 at (.866, 0.5). Using euclidian distance, the distance between them is 0.51; the distance between 12 and 6 (opposite sides of the wheel) is 2.0, which we can use to describe that that 12 and 1 are very closely related, while 12 and 6 are mathematically distant (opposites on the camelot wheel).

In [43]:
# song recommendations
songs = data.drop_duplicates(subset=["title_x", "artist_name"]).reset_index(drop=True)
features = songs.loc[:, "camelot_x":"loudness"].drop(["camelot_x", "camelot_y"], axis=1)
model = make_pipeline(StandardScaler(), NearestNeighbors()).fit(features)

# # use PCA for dimensionality reduction so that we can "graph" songs
coords = make_pipeline(StandardScaler(), PCA(n_components=2)).fit_transform(features)
songs["x"], songs["y"] = coords[:, 0], coords[:, 1]

alt.Chart(songs).encode(
    x=alt.X("x:Q", title=""),
    y=alt.Y("y:Q", title=""),
    tooltip=alt.Tooltip(data.columns.to_list()),
    color=alt.Color("energy:Q", scale=alt.Scale(scheme="redyellowblue", reverse=True))
).mark_point(filled=True, size=50, color="green").properties(width=800, height=600, title="PCA Songs").interactive()

alt.Chart(...)

In [15]:
# 5 songs closest to song 10 in our list (must standardize song first)
# TODO: this includes the song itself since it's its own neighbor
rec_distances, rec_indices = model[-1].kneighbors(model[:-1].transform(features.iloc[[10]]), 5)
recs = songs.iloc[*rec_indices]
recs

,request_id,request_time,mode,camelot,camelot_x,camelot_y,is_major,duration_x,tempo,popularity,...,release_genres,title_y,artist,album,duration_y,program_start_ts,plays,is_ad,x,y
10,01a15ddf-7063-73c6-86fb-6683e1eda112,2026-10-21 09:51:26.819852+00:00,major,3B,6.123234e-17,1.000000,1,152000,118,34,...,electronic | indie pop | indie rock | pop | ro...,Cash Wednesday,Skylar Spence,Prom King,149,2026-10-21 09:50:49+00:00,4,False,0.627476,0.270939
192,01a18090-eccc-7252-a321-c113ae9711b6,2026-10-28 03:32:23.884604+00:00,major,5B,-8.660254e-01,0.500000,1,139000,103,38,...,,Got Shocked,Wednesday,NaN,137,2026-10-28 03:31:26+00:00,1,False,0.860056,-0.573281
101,01a16ebf-53ad-714b-afff-c878bdbe7186,2026-10-24 16:29:54.989746+00:00,major,8B,-5.000000e-01,-0.866025,1,178000,108,47,...,,Picture Window,Japanese Breakfast,NaN,175,2026-10-24 16:29:17+00:00,15,False,0.637158,0.714359
196,01a1811f-c746-7137-a00d-e4822e7aaf4b,2026-10-28 06:08:25.926720+00:00,major,4B,-5.000000e-01,0.866025,1,156000,121,36,...,electronic | house | jazz | garage house,Machine Noise For A Quiet Daydream,Barry Can't Swim,Loner,156,2026-10-28 06:08:19+00:00,3,False,-0.072902,0.488680
344,01a1a16e-4851-7156-a5a3-d26e0ba47936,2026-11-03 12:42:01.681304+00:00,major,3B,6.123234e-17,1.000000,1,178000,126,35,...,rock,Somebody New,Tunde Adebimpe,NaN,177,2026-11-03 12:41:20+00:00,10,False,1.524205,0.450750


In [32]:
track_analysis_corr = track_analysis.corr(numeric_only=True)
track_analysis_corr_cols = track_analysis_corr.columns.to_list()

alt.Chart(track_analysis_corr).mark_circle(opacity=0.5, size=25).encode(
    x=alt.X(alt.repeat("column"), type="quantitative"),
    y=alt.Y(alt.repeat("row"), type="quantitative")
).properties(width=150, height=150).repeat(row=track_analysis_corr_cols, column=track_analysis_corr_cols)

alt.RepeatChart(...)

In [ ]:
# ignore perfectly correlated values on the diagonal
track_analysis_corr = track_analysis_corr.where(~np.eye(len(track_analysis_corr), dtype=bool))
# by itself, this returns duplicate pairs: track_analysis_corr.unstack().nlargest(5)... must first remove duplicates:
track_analysis_corr.unstack()[track_analysis_corr.unstack().index.map(lambda x: x[0] < x[1])].nlargest(2), track_analysis_corr.unstack()[track_analysis_corr.unstack().index.map(lambda x: x[0] < x[1])].nsmallest(2)

(energy  loudness     0.744979
         happiness    0.345585
 dtype: float64,
 acousticness  energy     -0.632251
               loudness   -0.498032
 dtype: float64)